In [1]:
# --- Import Necessary Libraries ---
import numpy as np
from scipy.stats import gamma as gamma_dist, norm, binom
from collections import Counter

# --- Phase 1, Step 1: Defining the 'Ground Truth' ---
np.random.seed(29)
z_true = ["lei", "sure", "or", "time"]
gamma_true = { "lei": 0.40, "sure": 0.40, "or": 0.16, "time": 0.50 }
n_features = 25
mu_true = {s: np.random.rand(n_features) for s in z_true}
variance = 0.1
sigma_true = {s: np.identity(n_features) * variance for s in z_true}
pi_bar_true = {
    "lei":  np.array([0.0, 1.0, 0.0, 0.0]),
    "sure": np.array([0.0, 0.0, 1.0, 0.0]),
    "or":   np.array([0.0, 0.0, 0.0, 1.0]),
    "time": np.array([0.0, 0.0, 0.0, 1.0]),
}

# --- Phase 1, Step 2: Generating Acoustic Datasets ---
def generate_acoustic_data(z_sequence, gamma_params, mu_params, sigma_params, rate, time_step_duration=0.01):
    acoustic_series = []
    for syllable in z_sequence:
        duration_in_seconds = gamma_params[syllable] / rate
        num_time_steps = max(1, round(duration_in_seconds / time_step_duration))
        for _ in range(num_time_steps):
            acoustic_series.append(np.random.multivariate_normal(mu_params[syllable], sigma_params[syllable]))
    return np.array(acoustic_series)

r_slow = 4.0
r_fast = 7.0
a_slow = generate_acoustic_data(z_true, gamma_true, mu_true, sigma_true, r_slow)
a_fast = generate_acoustic_data(z_true, gamma_true, mu_true, sigma_true, r_fast)

# --- Phase 2: Defining the Model and Hyperparameters ---
hyperparams = {
    'gamma': {'alpha': 2.0, 'beta': 5.0},
    'mu0': {'mu0': 0.5, 'tau0': 0.5},
    'sigma': {'a': 1.0, 'b': 10.0},
}

class RateHMM:
    def __init__(self, states, n_features, gamma_hyperparams, mu0_hyperparams, sigma_hyperparams, pi_bar_fixed):
        self.states = states
        self.state_map = {name: i for i, name in enumerate(states)}
        self.idx_to_state = {i: name for name, i in self.state_map.items()}
        self.n_states = len(states)
        self.n_features = n_features
        self.gamma_hyperparams = gamma_hyperparams
        self.mu0_hyperparams = mu0_hyperparams
        self.sigma_hyperparams = sigma_hyperparams
        self.pi_bar = {s: pi_bar_fixed[s] for s in states}
        self.gamma = {s: gamma_dist.rvs(a=self.gamma_hyperparams['alpha'], scale=1/self.gamma_hyperparams['beta']) for s in self.states}
        self.mu = {s: np.random.normal(loc=self.mu0_hyperparams['mu0'], scale=self.mu0_hyperparams['tau0'], size=n_features) for s in self.states}
        self.sigma_sq = {s: 1.0 / gamma_dist.rvs(a=self.sigma_hyperparams['a'], scale=1/self.sigma_hyperparams['b']) for s in self.states}

    def _get_transition_matrix(self, r):
        T = np.zeros((self.n_states, self.n_states))
        for i, s_from in enumerate(self.states):
            kappa = np.clip(self.gamma[s_from] / r, 0, 1)
            T[i, :] = (1 - kappa) * self.pi_bar[s_from]
            T[i, i] += kappa
        return T

    def _get_emission_log_probs(self, a_data):
        log_probs = np.zeros((len(a_data), self.n_states))
        for t, a_t in enumerate(a_data):
            for i, state in enumerate(self.states):
                log_probs[t, i] = norm.logpdf(a_t, loc=self.mu[state], scale=np.sqrt(self.sigma_sq[state])).sum()
        return log_probs

    def _viterbi_decode(self, a_data, r):
        """Finds the single most probable state sequence using the Viterbi algorithm."""
        T_matrix = self._get_transition_matrix(r)
        log_emissions = self._get_emission_log_probs(a_data)

        T = len(a_data)
        N = self.n_states

        viterbi_trellis = np.zeros((T, N))
        backpointers = np.zeros((T, N), dtype=int)

        # Initialization step
        viterbi_trellis[0, :] = -np.inf
        viterbi_trellis[0, self.state_map['lei']] = log_emissions[0, self.state_map['lei']]

        # Recursion step
        for t in range(1, T):
            for j in range(N):
                probs = viterbi_trellis[t-1, :] + np.log(T_matrix[:, j] + 1e-12)
                backpointers[t, j] = np.argmax(probs)
                viterbi_trellis[t, j] = np.max(probs) + log_emissions[t, j]

        # Termination step
        best_path = np.zeros(T, dtype=int)
        best_path[-1] = np.argmax(viterbi_trellis[-1, :])

        # Backtracking
        for t in range(T - 2, -1, -1):
            best_path[t] = backpointers[t + 1, best_path[t + 1]]

        return [self.idx_to_state[i] for i in best_path]

    def _sample_latent_states(self, a_data, r):
        """Samples a plausible state sequence using the Forward-Backward algorithm."""
        # This code remains the same but is now called *after* Viterbi pre-training
        T_matrix = self._get_transition_matrix(r)
        log_emissions = self._get_emission_log_probs(a_data)
        alpha = np.zeros((len(a_data), self.n_states))
        alpha[0, :] = -np.inf
        alpha[0, self.state_map['lei']] = log_emissions[0, self.state_map['lei']]
        for t in range(1, len(a_data)):
            for j in range(self.n_states):
                log_sum_exp = np.logaddexp.reduce(alpha[t-1, :] + np.log(T_matrix[:, j] + 1e-12))
                alpha[t, j] = log_sum_exp + log_emissions[t, j]
        z_sequence_indices = np.zeros(len(a_data), dtype=int)
        p = np.exp(alpha[-1, :] - np.logaddexp.reduce(alpha[-1, :]))
        z_sequence_indices[-1] = np.random.choice(self.n_states, p=p)
        for t in range(len(a_data) - 2, -1, -1):
            prev_z = z_sequence_indices[t+1]
            log_p = alpha[t, :] + np.log(T_matrix[:, prev_z] + 1e-12)
            p = np.exp(log_p - np.logaddexp.reduce(log_p))
            z_sequence_indices[t] = np.random.choice(self.n_states, p=p)
        return [self.idx_to_state[i] for i in z_sequence_indices]

    def _sample_emissions(self, a_data, z_sequence):
        tau0_sq = self.mu0_hyperparams['tau0']**2
        mu0 = self.mu0_hyperparams['mu0']
        a_sig = self.sigma_hyperparams['a']
        b_sig = self.sigma_hyperparams['b']
        for state in self.states:
            state_indices = [t for t, s in enumerate(z_sequence) if s == state]
            n_k = len(state_indices)
            if n_k == 0: continue
            a_k = a_data[state_indices]
            a_k_bar = a_k.mean(axis=0)
            var_post = 1 / (1/tau0_sq + n_k/self.sigma_sq[state])
            mean_post = var_post * (mu0/tau0_sq + (n_k * a_k_bar)/self.sigma_sq[state])
            self.mu[state] = np.random.normal(loc=mean_post, scale=np.sqrt(var_post))
            a_post = a_sig + (n_k * self.n_features) / 2
            b_post = b_sig + 0.5 * np.sum((a_k - self.mu[state])**2)
            self.sigma_sq[state] = 1.0 / gamma_dist.rvs(a=a_post, scale=1/b_post)

    def _sample_gamma(self, z_sequence, r, proposal_std=0.01):
        for state in self.states:
            current_gamma = self.gamma[state]
            n_self_transitions = 0
            n_total_from_state = 0
            for t in range(len(z_sequence) - 1):
                if z_sequence[t] == state:
                    n_total_from_state += 1
                    if z_sequence[t+1] == state:
                        n_self_transitions += 1
            if n_total_from_state == 0: continue
            proposed_gamma = np.random.normal(loc=current_gamma, scale=proposal_std)
            if proposed_gamma <= 0: continue
            current_kappa = np.clip(current_gamma / r, 1e-9, 1-1e-9)
            proposed_kappa = np.clip(proposed_gamma / r, 1e-9, 1-1e-9)
            log_lik_current = binom.logpmf(n_self_transitions, n_total_from_state, current_kappa)
            log_lik_proposed = binom.logpmf(n_self_transitions, n_total_from_state, proposed_kappa)
            log_prior_current = gamma_dist.logpdf(current_gamma, a=self.gamma_hyperparams['alpha'], scale=1/self.gamma_hyperparams['beta'])
            log_prior_proposed = gamma_dist.logpdf(proposed_gamma, a=self.gamma_hyperparams['alpha'], scale=1/self.gamma_hyperparams['beta'])
            log_acceptance_ratio = (log_lik_proposed + log_prior_proposed) - (log_lik_current + log_prior_current)
            if np.log(np.random.rand()) < log_acceptance_ratio:
                self.gamma[state] = proposed_gamma

    def fit(self, a_data, r, num_iterations=1000, pre_train_iterations=100):
        posterior_samples = []
        print(f"\n--- Starting MCMC Fitting for rate r={r} ---")
        for i in range(num_iterations):
            if i % 100 == 0: print(f"  Iteration {i}/{num_iterations}...")

            # Use Viterbi for pre-training, then switch to the sampler
            if i < pre_train_iterations:
                z_sequence = self._viterbi_decode(a_data, r)
            else:
                z_sequence = self._sample_latent_states(a_data, r)

            self._sample_emissions(a_data, z_sequence)
            self._sample_gamma(z_sequence, r)

            if i >= pre_train_iterations:
                posterior_samples.append({'z': z_sequence})
        print(f"--- MCMC Fitting Complete for rate r={r} ---")
        return posterior_samples

# --- Phase 3: Running Full Simulation and Analyzing Results ---
hmm_slow = RateHMM(states=z_true, n_features=n_features, gamma_hyperparams=hyperparams['gamma'], mu0_hyperparams=hyperparams['mu0'], sigma_hyperparams=hyperparams['sigma'], pi_bar_fixed=pi_bar_true)
posterior_samples_slow = hmm_slow.fit(a_slow, r_slow)
hmm_fast = RateHMM(states=z_true, n_features=n_features, gamma_hyperparams=hyperparams['gamma'], mu0_hyperparams=hyperparams['mu0'], sigma_hyperparams=hyperparams['sigma'], pi_bar_fixed=pi_bar_true)
posterior_samples_fast = hmm_fast.fit(a_fast, r_fast)

def get_most_probable_sequence(posterior_samples):
    if not posterior_samples: return ["No samples collected"]
    z_samples = [s['z'] for s in posterior_samples]
    num_time_steps = len(z_samples[0])
    most_probable_z = []
    for t in range(num_time_steps):
        time_step_states = [z[t] for z in z_samples]
        most_common_state = Counter(time_step_states).most_common(1)[0][0]
        most_probable_z.append(most_common_state)
    unique_sequence = []
    for state in most_probable_z:
        if not unique_sequence or unique_sequence[-1] != state:
            unique_sequence.append(state)
    return unique_sequence

inferred_sequence_slow = get_most_probable_sequence(posterior_samples_slow)
inferred_sequence_fast = get_most_probable_sequence(posterior_samples_fast)

# --- 4. Final Validation ---
print("\n\n--- FINAL RESULTS ---")
print(f"Ground Truth Sequence: {z_true}")
print("-" * 45)
print(f"Inferred Sequence (Fast Rate, r={r_fast}): {inferred_sequence_fast}")
print(f"Inferred Sequence (Slow Rate, r={r_slow}): {inferred_sequence_slow}")
print("-" * 45)
if "or" in inferred_sequence_fast and "or" not in inferred_sequence_slow:
    print("\n✅ Success! The model correctly perceived the function word 'or' in the fast context,")
    print("   but missed it in the slow context, explaining the 'disappearing word' phenomenon.")
else:
    print("\n❌ The simulation did not produce the expected result.")


--- Starting MCMC Fitting for rate r=4.0 ---
  Iteration 0/1000...
  Iteration 100/1000...
  Iteration 200/1000...
  Iteration 300/1000...
  Iteration 400/1000...
  Iteration 500/1000...
  Iteration 600/1000...
  Iteration 700/1000...
  Iteration 800/1000...
  Iteration 900/1000...
--- MCMC Fitting Complete for rate r=4.0 ---

--- Starting MCMC Fitting for rate r=7.0 ---
  Iteration 0/1000...
  Iteration 100/1000...
  Iteration 200/1000...
  Iteration 300/1000...
  Iteration 400/1000...
  Iteration 500/1000...
  Iteration 600/1000...
  Iteration 700/1000...
  Iteration 800/1000...
  Iteration 900/1000...
--- MCMC Fitting Complete for rate r=7.0 ---


--- FINAL RESULTS ---
Ground Truth Sequence: ['lei', 'sure', 'or', 'time']
---------------------------------------------
Inferred Sequence (Fast Rate, r=7.0): ['lei']
Inferred Sequence (Slow Rate, r=4.0): ['lei', 'sure', 'or']
---------------------------------------------

❌ The simulation did not produce the expected result.
